In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1g2lYetTr7-_0t-L7y0LPXyLh3GeFNDQr", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_01_intro.mp3"))

# 🚀 Plan Mode & Iterative Refinement — Execution Strategy Lab

In this notebook, we will simulate Claude Code's execution modes and practice the iterative refinement patterns tested on the exam. You will build a decision engine that recommends plan mode vs direct execution, and practice the four key refinement patterns: concrete examples, test-driven iteration, the interview pattern, and batching strategies.

By the end, you will be able to:
- Choose between plan mode and direct execution for any task
- Use the Explore subagent for codebase investigation
- Apply 4 refinement patterns: examples, test-driven, interview, batching
- Combine plan + direct execution in complex workflows

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/claude-code-workflows/practice/3/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

A developer asks Claude to "migrate the authentication system from session tokens to JWTs." Claude starts editing files immediately. Halfway through — 23 files changed — the developer realizes Claude chose the wrong JWT library, and the token refresh strategy is incompatible with the mobile app.

All 23 file changes need to be reverted.

This is the plan mode failure case. The developer should have used **plan mode** first — letting Claude investigate the codebase, propose an approach, and get approval before touching a single file. The exam tests your ability to make this judgment call.

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import json
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

# We'll simulate task analysis and mode selection

In [ ]:
#@title 🎧 Listen: Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition — The Decision Framework

Think of plan mode like this:

- **Direct execution** = You know exactly which aisle the item is in at the grocery store. Walk there, pick it up, done.
- **Plan mode** = You have a complex recipe with ingredients from multiple stores. You need to check inventory, compare prices, and plan your route before driving anywhere.

The key question: **Do I already know what files need to change and how?**

In [ ]:
#@title 🎧 Listen: Decision Engine Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_decision_engine_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
@dataclass
class TaskAnalysis:
    """Analyze a task to determine the best execution mode."""
    description: str
    estimated_files: int = 0
    scope_clear: bool = True
    multiple_approaches: bool = False
    architectural_impact: bool = False
    investigation_needed: bool = False

    @property
    def recommended_mode(self) -> str:
        """Determine the recommended execution mode."""
        # Score-based decision
        plan_score = 0

        if self.estimated_files > 10:
            plan_score += 2
        elif self.estimated_files > 5:
            plan_score += 1

        if not self.scope_clear:
            plan_score += 2

        if self.multiple_approaches:
            plan_score += 2

        if self.architectural_impact:
            plan_score += 2

        if self.investigation_needed:
            plan_score += 1

        if plan_score >= 3:
            return "plan_mode"
        elif self.investigation_needed:
            return "explore_then_decide"
        else:
            return "direct_execution"

    @property
    def reasoning(self) -> List[str]:
        reasons = []
        if self.estimated_files > 10:
            reasons.append(f"Large scope ({self.estimated_files} files)")
        if not self.scope_clear:
            reasons.append("Unclear scope — need investigation")
        if self.multiple_approaches:
            reasons.append("Multiple valid approaches exist")
        if self.architectural_impact:
            reasons.append("Architectural decision with cascading effects")
        if self.investigation_needed:
            reasons.append("Codebase investigation needed first")
        if not reasons:
            reasons.append("Clear scope, known approach")
        return reasons


# Test cases from the exam domain
tasks = [
    TaskAnalysis(
        description="Fix a typo in the README",
        estimated_files=1,
        scope_clear=True,
    ),
    TaskAnalysis(
        description="Add a new API endpoint for user preferences",
        estimated_files=4,
        scope_clear=True,
    ),
    TaskAnalysis(
        description="Migrate authentication from sessions to JWTs",
        estimated_files=45,
        scope_clear=False,
        multiple_approaches=True,
        architectural_impact=True,
    ),
    TaskAnalysis(
        description="Improve dashboard performance",
        estimated_files=0,  # Unknown
        scope_clear=False,
        investigation_needed=True,
    ),
    TaskAnalysis(
        description="Refactor the payment module to use Strategy pattern",
        estimated_files=15,
        multiple_approaches=True,
        architectural_impact=True,
    ),
    TaskAnalysis(
        description="Rename getUserById to findUserById across the codebase",
        estimated_files=12,
        scope_clear=True,
    ),
]

print("Task Analysis → Execution Mode Recommendation")
print("=" * 70)
for task in tasks:
    mode = task.recommended_mode.replace("_", " ").title()
    print(f"\n📋 {task.description}")
    print(f"   → {mode}")
    for reason in task.reasoning:
        print(f"     • {reason}")

In [ ]:
#@title 🎧 Listen: Stop Think
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_stop_think.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Notice the nuance in the last example: renaming a function across 12 files. Even though it touches many files, the scope is perfectly clear — find-and-replace with type checking. Direct execution is correct here because there is no ambiguity about the approach.

The exam tests this exact distinction: **file count alone does not determine the mode**. Scope clarity and approach ambiguity matter more.

---

### ✋ Stop and Think

What about "Add dark mode to the application"? Plan mode or direct execution?

*Answer: Plan mode. Even though "add dark mode" sounds simple, it involves multiple approaches (CSS variables vs theme context vs Tailwind dark mode), affects many files, and requires architectural decisions about where the theme state lives.*

In [ ]:
#@title 🎧 Listen: Explore Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_explore_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Explore → Decide → Execute Pattern

The Explore subagent is an intermediate step that prevents premature commitment.

In [ ]:
#@title 🎧 Listen: Explore Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_explore_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
@dataclass
class ExploreResult:
    """Simulated result from the Explore subagent."""
    files_found: List[str]
    patterns_identified: List[str]
    complexity_assessment: str
    recommended_next_step: str


def simulate_explore(query: str) -> ExploreResult:
    """Simulate the Explore subagent investigating a codebase."""

    scenarios = {
        "authentication": ExploreResult(
            files_found=[
                "src/middleware/auth.ts",
                "src/lib/session.ts",
                "src/api/login.ts",
                "src/api/logout.ts",
                "src/api/refresh.ts",
                "src/hooks/useAuth.ts",
                "src/context/AuthContext.tsx",
                "src/types/auth.ts",
                "tests/auth.test.ts",
                "config/auth.config.ts",
            ],
            patterns_identified=[
                "Session-based auth with express-session",
                "Redis for session storage",
                "Auth middleware on 34 API routes",
                "Mobile app uses cookie-based sessions",
                "Token refresh via /api/refresh endpoint",
            ],
            complexity_assessment="HIGH — 10 core files, 34 consuming routes, mobile app dependency",
            recommended_next_step="plan_mode",
        ),
        "performance": ExploreResult(
            files_found=[
                "src/components/Dashboard.tsx",
                "src/hooks/useDashboardData.ts",
                "src/api/dashboard.ts",
            ],
            patterns_identified=[
                "Dashboard re-renders on every state change (no memoization)",
                "3 separate API calls that could be batched",
                "Large list rendering without virtualization",
            ],
            complexity_assessment="MEDIUM — 3 files, clear bottlenecks identified",
            recommended_next_step="direct_execution",
        ),
    }

    for key, result in scenarios.items():
        if key in query.lower():
            return result

    return ExploreResult(
        files_found=["(no matching files found)"],
        patterns_identified=["(no patterns identified)"],
        complexity_assessment="UNKNOWN",
        recommended_next_step="investigate_further",
    )


# Simulate exploring before deciding
print("Explore → Decide → Execute Workflow")
print("=" * 60)

for query in ["authentication system", "performance bottlenecks"]:
    result = simulate_explore(query)
    print(f"\n🔍 Explore: '{query}'")
    print(f"   Files found: {len(result.files_found)}")
    for f in result.files_found[:5]:
        print(f"     📄 {f}")
    if len(result.files_found) > 5:
        print(f"     ... and {len(result.files_found) - 5} more")
    print(f"   Patterns:")
    for p in result.patterns_identified[:3]:
        print(f"     • {p}")
    print(f"   Complexity: {result.complexity_assessment}")
    print(f"   → Next step: {result.recommended_next_step.replace('_', ' ').title()}")

This is the Explore subagent's value: it does verbose investigation **in an isolated context** so your main conversation stays clean. After exploring, you make an informed decision about plan mode vs direct execution.

In [ ]:
#@title 🎧 Listen: Refinement Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_refinement_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Iterative Refinement Patterns

Now let us practice the four refinement patterns the exam tests.

In [ ]:
#@title 🎧 Listen: Examples Pattern
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_examples_pattern.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Pattern 1: Concrete Input/Output Examples

When you need consistent transformations, provide 2-3 examples to anchor Claude's behavior.

In [ ]:
def demonstrate_example_pattern():
    """Show how concrete examples eliminate ambiguity."""

    # BAD: Vague instruction
    vague_prompt = "Convert these functions to async/await style."

    # GOOD: Concrete examples
    example_prompt = """Transform these function signatures from callback style to async/await:

Example 1:
  Input:  function getUser(id, callback) { db.find(id, callback); }
  Output: async function getUser(id): Promise<User> { return db.find(id); }

Example 2:
  Input:  function saveOrder(order, onSuccess, onError) { ... }
  Output: async function saveOrder(order): Promise<Order> { ... }

Now transform all functions in src/legacy/api.ts."""

    print("❌ VAGUE PROMPT:")
    print(f"   '{vague_prompt}'")
    print("   Problem: Ambiguous. Does 'async/await style' mean Promises?")
    print("   Callbacks? What about error handling?\n")

    print("✅ EXAMPLE-ANCHORED PROMPT:")
    print(example_prompt)
    print("\n   Result: Claude follows the exact pattern shown in examples.")
    print("   Key insight: 2-3 examples > 10 lines of verbal description.")


demonstrate_example_pattern()

In [ ]:
#@title 🎧 Listen: Tdd Pattern
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_tdd_pattern.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Pattern 2: Test-Driven Iteration

Write tests first. Let Claude iterate on the implementation until tests pass.

In [ ]:
def simulate_test_driven_iteration():
    """Simulate the test-driven refinement loop."""

    # Define the test suite (the CONTRACT)
    test_cases = [
        {"input": "cache.get('user:123')", "expected": "returns None for missing keys"},
        {"input": "cache.set('user:123', data, ttl=300)", "expected": "stores with 5-min TTL"},
        {"input": "cache.get('user:123')", "expected": "returns data before TTL expires"},
        {"input": "time.sleep(301); cache.get('user:123')", "expected": "returns None after TTL"},
        {"input": "cache.delete('user:123')", "expected": "removes the key"},
        {"input": "cache.clear()", "expected": "removes all keys"},
    ]

    print("Test-Driven Iteration Workflow")
    print("=" * 55)
    print("\nStep 1: Write tests FIRST (define the contract)")
    print("-" * 55)
    for i, tc in enumerate(test_cases, 1):
        print(f"  Test {i}: {tc['input']}")
        print(f"          → {tc['expected']}")

    # Simulate iterations
    iterations = [
        {"passed": 3, "failed": 3, "issue": "TTL not implemented"},
        {"passed": 5, "failed": 1, "issue": "clear() doesn't reset TTL timers"},
        {"passed": 6, "failed": 0, "issue": None},
    ]

    print("\nStep 2: Claude iterates until all tests pass")
    print("-" * 55)
    for i, it in enumerate(iterations, 1):
        status = "✅ ALL PASS" if it["failed"] == 0 else f"❌ {it['failed']} failing"
        print(f"  Iteration {i}: {it['passed']}/{len(test_cases)} passed — {status}")
        if it["issue"]:
            print(f"              Fix: {it['issue']}")

    print("\n✅ Implementation complete in 3 iterations!")
    print("   Key insight: Tests give Claude specific, actionable feedback")
    print("   on each iteration. Much better than 'try again'.")


simulate_test_driven_iteration()

In [ ]:
#@title 🎧 Listen: Interview Pattern
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_interview_pattern.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Pattern 3: The Interview Pattern

For unfamiliar domains, instruct Claude to ask questions before implementing.

In [ ]:
def simulate_interview_pattern():
    """Simulate the interview pattern for requirement discovery."""

    user_request = "Add rate limiting to our API."

    # What Claude WOULD ask before implementing
    interview_questions = [
        {
            "question": "Which endpoints need rate limiting?",
            "options": ["All endpoints", "Only public endpoints", "Specific endpoints (list them)"],
            "impact": "Determines scope of middleware configuration"
        },
        {
            "question": "What should the rate limits be?",
            "options": ["100 req/min per user", "1000 req/min per IP", "Custom per endpoint"],
            "impact": "Determines complexity of limit configuration"
        },
        {
            "question": "How should limits be tracked?",
            "options": ["In-memory (single server)", "Redis (distributed)", "Database"],
            "impact": "Architecture: single-server vs distributed system"
        },
        {
            "question": "What should happen when a limit is exceeded?",
            "options": ["429 with retry-after header", "Queue the request", "Degrade gracefully"],
            "impact": "User experience and error handling strategy"
        },
        {
            "question": "Do you need different limits for authenticated vs anonymous users?",
            "options": ["Yes, higher for authenticated", "No, same limits", "Tiered by subscription plan"],
            "impact": "Integration with auth system complexity"
        },
    ]

    print("Interview Pattern Simulation")
    print("=" * 60)
    print(f"\nUser request: '{user_request}'")
    print("\nWithout interview pattern:")
    print("  Claude assumes defaults → implements in-memory rate limiting")
    print("  → Breaks in production (multiple servers, no shared state)")
    print("  → Rework required ❌")

    print("\nWith interview pattern — Claude asks first:")
    print("-" * 60)
    for i, q in enumerate(interview_questions, 1):
        print(f"\n  Q{i}: {q['question']}")
        for opt in q['options']:
            print(f"      • {opt}")
        print(f"      Impact: {q['impact']}")

    print("\n✅ After answers: Claude implements exactly what you need.")
    print("   Key insight: 5 minutes of questions saves hours of rework.")


simulate_interview_pattern()

In [ ]:
#@title 🎧 Listen: Batching Pattern
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_batching_pattern.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Pattern 4: Batching Interacting vs Independent Issues

In [ ]:
def demonstrate_batching():
    """Show when to batch related changes vs submit independently."""

    # Interacting changes — BATCH TOGETHER
    interacting = {
        "scenario": "Three changes to the User model",
        "changes": [
            "Add emailVerified boolean field",
            "Update registration to send verification email",
            "Add middleware blocking unverified users from protected routes",
        ],
        "why_batch": "All three affect the User model and auth flow. "
                     "Implementing #1 without #3 leaves a security gap. "
                     "Implementing #3 without #1 breaks the build.",
        "strategy": "BATCH — submit all three in one prompt",
    }

    # Independent changes — SUBMIT SEQUENTIALLY
    independent = {
        "scenario": "Three unrelated fixes",
        "changes": [
            "Fix CSS alignment on the login page",
            "Add pagination to the /api/products endpoint",
            "Update the README with new environment variables",
        ],
        "why_separate": "These touch different files with no shared interfaces. "
                       "Each can be reviewed and tested independently.",
        "strategy": "SEQUENTIAL — submit one at a time for focused attention",
    }

    print("Batching Strategy")
    print("=" * 60)

    print(f"\n📦 BATCH TOGETHER: {interacting['scenario']}")
    for i, change in enumerate(interacting['changes'], 1):
        print(f"   {i}. {change}")
    print(f"   Why: {interacting['why_batch']}")
    print(f"   → {interacting['strategy']}")

    print(f"\n📝 SUBMIT SEPARATELY: {independent['scenario']}")
    for i, change in enumerate(independent['changes'], 1):
        print(f"   {i}. {change}")
    print(f"   Why: {independent['why_separate']}")
    print(f"   → {independent['strategy']}")

    print("\n💡 Rule of thumb: If changes share data structures,")
    print("   interfaces, or state — batch them. Otherwise, separate.")


demonstrate_batching()

In [ ]:
#@title 🎧 Listen: Todo Mode
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_todo_mode.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. 🔧 Your Turn — Mode Selection Practice

In [ ]:
# ============ TODO ============
# For each task, choose: "plan", "direct", or "explore_first"
# Then provide one reason for your choice.

exam_tasks = {
    "Upgrade React from v18 to v19 across the entire app": "???",
    "Fix a null pointer exception in getUserProfile()": "???",
    "Add a caching layer to the API": "???",
    "Update the copyright year in the footer": "???",
    "Refactor the monolith into microservices": "???",
    "Add a loading spinner to the search results page": "???",
}
# ==============================

In [ ]:
# ✅ Verification
correct = {
    "Upgrade React from v18 to v19 across the entire app": "plan",
    "Fix a null pointer exception in getUserProfile()": "direct",
    "Add a caching layer to the API": "plan",
    "Update the copyright year in the footer": "direct",
    "Refactor the monolith into microservices": "plan",
    "Add a loading spinner to the search results page": "direct",
}

explanations = {
    "Upgrade React from v18 to v19 across the entire app": "Many files, breaking changes, multiple migration paths → Plan mode",
    "Fix a null pointer exception in getUserProfile()": "Clear scope, known location, single fix → Direct execution",
    "Add a caching layer to the API": "Multiple approaches (Redis, in-memory, CDN), architecture decision → Plan mode",
    "Update the copyright year in the footer": "Trivial change, one file → Direct execution",
    "Refactor the monolith into microservices": "Massive scope, architectural decisions, many approaches → Plan mode",
    "Add a loading spinner to the search results page": "Clear scope, single component, known pattern → Direct execution",
}

score = 0
for task, answer in exam_tasks.items():
    expected = correct[task]
    if answer.strip().lower() == expected:
        print(f"✅ Correct! {explanations[task]}")
        score += 1
    else:
        print(f"❌ Expected '{expected}'. {explanations[task]}")

print(f"\nScore: {score}/{len(exam_tasks)}")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Reflection and Next Steps

**Key takeaways:**
1. **Plan mode**: Large scope, unclear approach, architectural decisions, multiple valid paths
2. **Direct execution**: Clear scope, known approach, single-file or mechanical changes
3. **Explore subagent**: Investigate first in an isolated context, then decide on mode
4. **Concrete examples**: 2-3 input/output examples beat paragraphs of description
5. **Test-driven**: Write tests first, iterate on failures — gives Claude specific feedback
6. **Interview pattern**: For unfamiliar domains, ask questions before implementing
7. **Batching**: Batch interacting changes together, submit independent changes separately

**Exam tip:** The exam often presents borderline cases. When in doubt, lean toward plan mode — the cost of planning is low, but the cost of rework is high.